In [55]:
from pathlib import Path
import numpy as np
import geopandas as gpd
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon
import contextily as cx
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)

In [56]:
# Gevraagd wordt een dataframe voor de UNPAVED zoals dit:
#	    id	    total_area	lu_areas	surface_level	soiltype	surface_storage	infiltration_capacity	initial_gwd	meteo_area	px	py	boundary_node
# code												
# 15.0	15.0	1375	250 0 0 0 0 0 0 0 0 0 225 0 0 0 0 0	                16.93   107	10.000	100.000	1.20	15.0	199378	395163	lat_15.0
# 55.0	55.0	303875	124200 18000 0 0 0 0 0 0 0 150 68125 0 11875 0...	21.69	105	10.000	100.000	1.20	55.0	197488	392239	lat_55.0
# 56.0	56.0	13300	5400 0 0 0 0 0 0 0 0 0 4425 0 375 0 1150 0	        20.63	113	10.000	100.000	1.20	56.0	197789	392200	lat_56.0
# 57.0	57.0	60925	6550 725 0 22800 0 0 0 0 0 0 9375 0 4600 0 175 0	21.49	113	10.000	100.000	1.20	57.0	197982	392247	lat_57.0

# en een dataframe voor ernst zoals deze:
# 	    id	    cvo	            lv	        cvi	    cvs
# code					
# 15.0	15.0	300 2000 100000	0.0 1.0 2.0	300.00	5.00
# 55.0	55.0	300 2000 100000	0.0 1.0 2.0	300.00	5.00
# 56.0	56.0	300 2000 100000	0.0 1.0 2.0	300.00	5.00
# 57.0	57.0	300 2000 100000	0.0 1.0 2.0	300.00	5.00

In [57]:
# For land use and soil type a coding is prescribed. For landuse, the legend of the map is expected to be as follows: <br>
landuse_mapping = {
    "potatoes": 1,
    "wheat": 2,
    "sugar beet": 3,
    "corn": 4,
    "other crops": 5,
    "bulbous plants": 6,
    "orchard": 7,
    "grass": 8,
    "deciduous forest": 9,
    "coniferous forest": 10,
    "nature": 11,
    "barren": 12,
    "open water": 13,
    "built-up": 14,
    "greenhouses": 15
}

# For classes 1-12, the areas are calculated from the provided raster and remapped to the classification in the Sobek RR-tables.


# The coding for the soil types:<br>
soiltype_first_mapping = {
    "podzol": "Podzol (grof zand)"
}

soiltype_mapping = {
    "Veengrond met veraarde bovengrond": 1,
    "Veengrond met veraarde bovengrond, zand": 2,
    "Veengrond met kleidek": 3,
    "Veengrond met kleidek op zand": 4,
    "Veengrond met zanddek op zand": 5,
    "Veengrond op ongerijpte klei": 6,
    "Stuifzand": 7,
    "Podzol (Leemarm, fijn zand)": 8,
    "Podzol (zwak lemig, fijn zand)": 9,
    "Podzol (zwak lemig, fijn zand op grof zand)": 10,
    "Podzol (lemig keileem)": 11,
    "Enkeerd (zwak lemig, fijn zand)": 12,
    "Beekeerd (lemig fijn zand)": 13,
    "Podzol (grof zand)": 14,
    "Zavel": 15,
    "Lichte klei": 16,
    "Zware klei": 17,
    "Klei op veen": 18,
    "Klei op zand": 19,
    "Klei op grof zand": 20,
    "Leem": 21
}

# And surface elevation needs to be in m+NAP.

In [58]:
def generate_unpaved_df_from_rr_input(gdf):
    df_unpaved = pd.DataFrame()
    df_unpaved["code"] = "unpaved_" + gdf["GFEIDENT"]
    df_unpaved["id"] = "unpaved_" + gdf["GFEIDENT"]
    df_unpaved["total_area"] = gdf["Area_RR_unpaved_m2"].astype(int)
    df_unpaved["lu_areas"] = gdf["Area_RR_unpaved_m2"].astype(int).astype(str) + " 0"*15
    df_unpaved["surface_level"] = gdf["SurfaceLevel_mNAP"]
    df_unpaved["soiltype"] = gdf["CapSimSoilType"].map(soiltype_first_mapping).map(soiltype_mapping) + 100              # soiltype_mapping + 100
    df_unpaved["surface_storage"] = gdf["StorageOnLand_mm"]
    df_unpaved["infiltration_capacity"] = gdf["InfiltrationCapacity_mmph"]
    df_unpaved["initial_gwd"] = gdf["InitialGroundwaterLevel_mBelowSurface"]
    df_unpaved["meteo_area"] = gdf["MeteoStationName"]
    df_unpaved["px"] = gdf.geometry.x
    df_unpaved["py"] = gdf.geometry.y
    df_unpaved["boundary_node"] = "lateral_" + gdf["GFEIDENT"].astype(str)
    df_unpaved = df_unpaved.set_index("code")
    return df_unpaved


def generate_ernst_df_from_rr_input(gdf):
    df_ernst = pd.DataFrame()
    df_ernst["code"] = "ernst_" + gdf["GFEIDENT"].astype(str)
    df_ernst["id"] = "ernst_" + gdf["GFEIDENT"].astype(str)
    df_ernst["cvo"] = gdf.apply(lambda row: " ".join([str(row["FirstDrainResistance_d"]), str(row["SecondDrainResistance_d"]), str(row["ThirdDrainResistance_d"])]), axis=1)
    gdf["FirstDrainLevel_m"] = (gdf["SurfaceLevel_mNAP"] - gdf["FirstDrainLevel_mNAP"]).round(2)
    gdf["SecondDrainLevel_m"] = (gdf["SurfaceLevel_mNAP"] - gdf["SeconDrainLevel_mNAP"]).round(2)
    gdf["ThirdDrainLevel_m"] = (gdf["SurfaceLevel_mNAP"] - gdf["ThirdDrainLevel_mNAP"]).round(2)
    df_ernst["lv"] = gdf.apply(lambda row: " ".join([str(row["FirstDrainLevel_m"]), str(row["SecondDrainLevel_m"]), str(row["ThirdDrainLevel_m"])]), axis=1)
    df_ernst["cvi"] = gdf["OpenWaterHorizontalInflowResistance_d"].astype(str)
    df_ernst["cvs"] = gdf["SurfaceOverlandFlowResistance_d"].astype(str)
    df_ernst = df_ernst.set_index("code")
    return df_ernst


def generate_rr_unpaved_ernst_from_input(dir_scenario_input, dir_scenario_output):
    seasons = ["zomer", "winter"]
    gpkg_rr_input_zomer_winter = ["RR_input_ZOMER.gpkg", "RR_input_WINTER.gpkg"]

    gdf_rr_input = {}

    for season, gpkg_rr_input in zip(seasons, gpkg_rr_input_zomer_winter):
        print(f"- season: {season}")

        gdf = gpd.read_file(dir_scenario_input / gpkg_rr_input)

        df_ernst = generate_ernst_df_from_rr_input(gdf)
        df_unpaved = generate_unpaved_df_from_rr_input(gdf)
        gdf_unpaved = gpd.GeoDataFrame(df_unpaved, geometry=gpd.points_from_xy(df_unpaved.px, df_unpaved.py), crs=28992)

        gdf_rr_input[season] = {}
        gdf_rr_input[season]["input"] = gdf
        gdf_rr_input[season]["ernst"] = df_ernst
        gdf_rr_input[season]["unpaved"] = gdf_unpaved

        df_ernst.to_csv(dir_scenario_output / f"df_ernst_{season}.csv")
        df_unpaved.to_csv(dir_scenario_output / f"df_unpaved_{season}.csv")
        gdf_unpaved.to_file(dir_scenario_output / f"gdf_unpaved_{season}.gpkg", layer=f"gdf_unpaved_{season}", driver="GPKG")
    
    return gdf_rr_input   

In [59]:
from datetime import timedelta

def write_bui(df, outfile, timestep_seconds=3600):
    stations = list(df.columns)
    nstations = len(stations)

    start = df.index[0]
    end = df.index[-1]

    duration = end - start + timedelta(seconds=timestep_seconds)

    dd = duration.days
    hh, rem = divmod(duration.seconds, 3600)
    mm, ss = divmod(rem, 60)

    with open(outfile, "w") as f:
        # Header
        f.write(f"*Name of this file: {outfile}\n")
        f.write("*Date and time of construction: 00/00/2000 00:00:00.\n")
        f.write("1\n")
        f.write("*Aantal stations\n")
        f.write(f"{nstations}\n")
        f.write("*Namen van stations\n")

        for s in stations:
            f.write(f"'{s}'\n")

        f.write("*Aantal gebeurtenissen (omdat het 1 bui betreft is dit altijd 1)\n")
        f.write("*en het aantal seconden per waarnemingstijdstap\n")
        f.write(f"1 {timestep_seconds}\n")
        f.write("*Elke commentaarregel wordt begonnen met een * (asterisk).\n")
        f.write("*Eerste record bevat startdatum en -tijd, lengte van de gebeurtenis in dd hh mm ss\n")
        f.write("*Het format is: yyyymmdd:hhmmss:ddhhmmss\n")
        f.write("*Daarna voor elk station de neerslag in mm per tijdstap.\n")

        # Startrecord
        f.write(
            f"{start.year} {start.month} {start.day} "
            f"{start.hour} {start.minute} {start.second} "
            f"{dd} {hh} {mm} {ss}\n"
        )

        # Tijdstappen
        for _, row in df.iterrows():
            line = " ".join(f"{v:.3f}" for v in row.values)
            f.write(line + "\n")

#### BASIS DIRECTORIES

In [60]:
# INPUT vanuit WRIJ voor RR unpaved methode
dir_data = Path("..\\..\\WRIJ_RR_Unpaved_methode_01_data\\")
dir_model_input = Path("..\\..\\WRIJ_RR_Unpaved_methode_02_tussenresultaat")

In [61]:
dir_basis_data = dir_model_input / "basisdata"

gebieden_path = dir_basis_data / "gebieden.gpkg"
path_watergang = dir_basis_data / "watergang.gpkg"
path_laterale_knoop = dir_basis_data / "laterale_knoop.gpkg"
path_afwateringseenheden = dir_basis_data / "afwateringseenheden.gpkg"

watergang = gpd.read_file(path_watergang)
laterale_knoop = gpd.read_file(path_laterale_knoop)
afwateringseenheden = gpd.read_file(path_afwateringseenheden)
afwateringseenheden["code"] = afwateringseenheden["GFEIDENT"].astype(str)
afwateringseenheden["globalid"] = afwateringseenheden["GLOBALID"].astype(str)
afwateringseenheden["lateraleknoopid"] = "lateral_" + afwateringseenheden["code"].astype(str)
afwateringseenheden = afwateringseenheden[["code", "globalid", "lateraleknoopid", "geometry"]]

gebieden = gpd.read_file(gebieden_path, layer="gebieden")

#### PREPARE INPUT FOR COMPLETE AREA

In [62]:
# INPUT vanuit WRIJ voor RR unpaved methode
dir_scenarios_data = Path(dir_data, "rr_scenarios_data")
dir_scenarios_input = Path(dir_model_input, "rr_scenarios_input")

scenario_gdf_input = {}

list_scenarios = [p.name for p in list(dir_scenarios_data.iterdir())]
list_scenarios = [s for s in list_scenarios if s.startswith("scenario")]

for scenario in list_scenarios:
    print(f"Scenario: {scenario}")
    dir_scenario_input = dir_scenarios_data / scenario
    dir_scenario_output = dir_scenarios_input / scenario
    
    gdfs_rr_input = generate_rr_unpaved_ernst_from_input(dir_scenario_input, dir_scenario_output)
    scenario_gdf_input[scenario] = gdfs_rr_input

Scenario: scenario_test
- season: zomer
- season: winter


#### PREPARE INPUT FOR PROJECT/PILOT AREAS

In [63]:
list_scenarios = list(scenario_gdf_input.keys())
seasons = scenario_gdf_input[list_scenarios[0]].keys()

for i, gebied in gebieden.iterrows():
    print(gebied.Pilotgebied_ID)
    dir_input_gebied = dir_model_input / "rr_scenarios_input_gebied" / f"gebied_{gebied.Pilotgebied_ID}"
    if not dir_input_gebied.exists():
        dir_input_gebied.mkdir(parents=True, exist_ok=True)
    
    # watergang
    watergang_gebied = watergang.clip(gebied.geometry).explode()
    watergang_gebied.to_file(dir_input_gebied / f"watergang.gpkg", layer=f"watergang", driver="GPKG")

    for scenario in list_scenarios:
        print(scenario)
        dir_input_gebied_scenario = dir_input_gebied / scenario
        if not dir_input_gebied_scenario.exists():
            dir_input_gebied_scenario.mkdir(parents=True, exist_ok=True)
        
        for season in seasons:
            print(season)
            # rr_unpaved
            scenario_gdf_unpaved_gebied = scenario_gdf_input[scenario][season]["unpaved"].clip(gebied.geometry)
            scenario_gdf_unpaved_gebied.to_file(dir_input_gebied_scenario / f"gdf_unpaved_{season}.gpkg", layer=f"gdf_unpaved_{season}", driver="GPKG")

            # rr_unpaved
            scenario_df_unpaved_gebied = scenario_gdf_unpaved_gebied.drop(columns="geometry")
            scenario_df_unpaved_gebied.to_csv(dir_input_gebied_scenario / f"df_unpaved_{season}.csv")

            # rr_ernst
            sel_ernst = scenario_gdf_input[scenario][season]["ernst"].index.str.replace("ernst_", "unpaved_")
            scenario_df_ernst_gebied = scenario_gdf_input[scenario][season]["ernst"].loc[sel_ernst.isin(scenario_gdf_unpaved_gebied.index)]
            scenario_df_ernst_gebied.to_csv(dir_input_gebied_scenario / f"df_ernst_{season}.csv")

    list_afw_eenheden = list(scenario_gdf_unpaved_gebied.index.str.replace("unpaved_", ""))

    # afwateringseenheden
    afwateringseenheden_gebied = afwateringseenheden[afwateringseenheden["code"].isin(list_afw_eenheden)]
    afwateringseenheden_gebied.to_file(dir_input_gebied / f"afwateringseenheden.gpkg", layer=f"afwateringseenheden", driver="GPKG")

    # laterale knopen
    laterale_knoop_gebied = laterale_knoop[laterale_knoop["code"].str.replace("lateral_", "").isin(list_afw_eenheden)]
    laterale_knoop_gebied.to_file(dir_input_gebied / f"laterale_knoop.gpkg", layer=f"laterale_knoop", driver="GPKG")

0
scenario_test
zomer
winter
1
scenario_test
zomer
winter
2
scenario_test
zomer
winter
3
scenario_test
zomer
winter


#### METEO DATA

In [64]:
start_date = "2010-4-1"
end_date = "2018-12-31"
cut_period_month = 12

In [65]:
# METEO - Verdamping
dir_meteo = Path(dir_scenarios_data, "meteo")
file_verdamping = "verdamping_hupsel.xlsx"

verdamping = pd.read_excel(dir_meteo / file_verdamping, index_col=0, parse_dates=True)
verdamping.columns = ["verdamping"]

verdamping["jaar"] = verdamping.index.year
verdamping["maand"] = verdamping.index.month
verdamping["dag"] = verdamping.index.day
verdamping = verdamping[["jaar", "maand", "dag", "verdamping"]]

verdamping = verdamping.loc[start_date:end_date]

header_verdamping = (
    "*Verdampingsfile verdamping Hupsel\n"
    "*Meteo data: evaporation intensity in mm/day\n"
    "*First record: start date, data in mm/day\n"
    "*Datum (year month day), verdamping (mm/dag) voor elk weerstation\n"
    "*jaar maand dag verdamping[mm]\n"
)

output_file = Path(dir_scenarios_input, "meteo", "METEO_VERDAMPING.EVP")

with open(output_file, "w") as f:
    f.write(header_verdamping)
    verdamping.to_string(
        f,
        index=False,
        header=False,
        formatters={"verdamping": "{:.3f}".format}
    )

verdamping

,jaar,maand,dag,verdamping
YYYYMMDD,,,,
2010-04-01,2010,4,1,1.4
2010-04-02,2010,4,2,2.2
2010-04-03,2010,4,3,1.2
2010-04-04,2010,4,4,1.3
2010-04-05,2010,4,5,2.1
...,...,...,...,...
2018-12-27,2018,12,27,0.3
2018-12-28,2018,12,28,0.1
2018-12-29,2018,12,29,0.1


In [66]:
# METEO - Neerslag
dir_neerslag_data = Path(dir_scenarios_data, "meteo", "neerslag_tijdreeksen\\output_tijdreeksen")

meteo_stations = scenario_gdf_input[list_scenarios[0]]["zomer"]["input"]["MeteoStationName"].unique()
neerslag_tijdseries = pd.DataFrame()

for meteo_station in meteo_stations:
    # print(meteo_station)
    path_tijdserie = Path(dir_neerslag_data, meteo_station + ".txt")
    if path_tijdserie.exists():
        tijdserie = pd.read_csv(path_tijdserie, sep=";", index_col=0, parse_dates=["YYYYMMDDHH"], date_format="%Y%m%d%H")
        neerslag_tijdseries[meteo_station] = tijdserie

neerslag_tijdseries = neerslag_tijdseries.loc[start_date:end_date]

write_bui(
    neerslag_tijdseries, 
    Path(dir_scenarios_input, "meteo", "METEO_NEERSLAG.BUI"), 
    timestep_seconds=3600
)

neerslag_tijdseries

,245725_440793
YYYYMMDDHH,
2010-04-01 00:00:00,0.00
2010-04-01 01:00:00,0.53
2010-04-01 02:00:00,0.16
2010-04-01 03:00:00,0.00
2010-04-01 04:00:00,0.00
...,...
2018-12-31 19:00:00,0.00
2018-12-31 20:00:00,0.00
2018-12-31 21:00:00,0.00


In [67]:
list_scenarios = list(scenario_gdf_input.keys())
seasons = scenario_gdf_input[list_scenarios[0]].keys()

for i, gebied in gebieden.iterrows():
    print(gebied.Pilotgebied_ID)
    dir_input_gebied = dir_model_input / "rr_scenarios_input_gebied" / f"gebied_{gebied.Pilotgebied_ID}"
    dir_input_gebied_meteo = dir_input_gebied / "meteo"
    if not dir_input_gebied_meteo.exists():
        dir_input_gebied_meteo.mkdir(parents=True, exist_ok=True)
    
    for scenario in list_scenarios:
        for season in seasons:
            # meteo
            scenario_gdf_input_gebied = scenario_gdf_input[scenario][season]["input"].clip(gebied.geometry)
            meteo_stations_gebied = list(scenario_gdf_input_gebied["MeteoStationName"].unique())
            neerslag_tijdseries_gebied = neerslag_tijdseries[meteo_stations_gebied]
            write_bui(
                neerslag_tijdseries_gebied, 
                dir_input_gebied_meteo / "METEO_NEERSLAG.BUI", 
                timestep_seconds=3600
            )

            verdamping_file = Path(dir_input_gebied_meteo, "METEO_VERDAMPING.EVP")

            with open(verdamping_file, "w") as f:
                f.write(header_verdamping)
                verdamping.to_string(
                    f,
                    index=False,
                    header=False,
                    formatters={"verdamping": "{:.3f}".format}
                )

            verdamping
            break
        break

0
1
2
3


In [68]:
# GEBIED
for i, gebied in gebieden.iterrows():
    dir_input_gebied = dir_model_input / "rr_scenarios_input_gebied" / f"gebied_{gebied.Pilotgebied_ID}"

    gebied_gdf = gebieden.iloc[[i]]
    gebied_gdf.to_file(dir_input_gebied / f"gebied.gpkg", layer=f"gebied", driver="GPKG")